# 拓展：归因模型深度拓展——从"选模型"到"做决策"

> **本拓展新增 10 个主课程未覆盖的深度概念**，其中 4 个来自全课程永久空白区，代码/数学深度占比 100%
> **配对主课**：W3D1 周一_归因模型.ipynb | **建议用时**：2-2.5h
> **阅读指引**：按顺序学习可获得最佳体验——每个概念群标注了关系类型（前置/并行/防护/落地/替代），帮助理解概念之间的依赖关系

---

## 知识网络预览

```
                         ┌─────────────────────────────┐
                         │   主课核心：归因六模型       │
                         │   末次/首次/线性/衰减/       │
                         │   位置/Shapley               │
                         └───────────┬─────────────────┘
                                     │
         ┌───────────────────────────┼───────────────────────────┐
         │                           │                           │
    ┌────▼──────────┐         ┌──────▼──────┐           ┌───────▼──────┐
    │ [前置] 必修    │         │ [并行] 必修   │           │ [替代] 选修   │
    │ ①归因窗口量化  │         │ ②边际ROI vs   │           │ ③MMM营销组合  │
    │   选择 [M][C]  │         │   平均ROI [P0] │           │   建模 [M][B]  │
    └────┬──────────┘         └──────┬──────┘           └───────┬──────┘
         │                           │                           │
         │              ┌────────────┼────────────┐              │
         │              │            │            │              │
    ┌────▼──────────┐ ┌─▼──────────┐ ┌▼──────────┐ ┌▼──────────┐ ┌▼──────────┐
    │ [防护] 必修    │ │[防护] 必修  │ │[防护] 选修 │ │[落地] 必修 │ │[替代] 选修 │
    │ ④归因中辛普森  │ │⑤归因模型   │ │⑥Bootstrap  │ │⑦渠道协同   │ │⑧预算重分配 │
    │   悖论 [M][C]  │ │  稳健性检验 │ │  权重CI [P1]│ │  效应量化   │ │  沙盘 [P0]  │
    └───────┬───────┘ └─────┬──────┘ └─────┬──────┘ └─────┬─────┘ └─────┬─────┘
            │               │              │              │             │
            └───────────────┴──────────────┴──────────────┴─────────────┘
                                    │
                           ┌────────▼────────┐
                           │ [选修] 自诊深化  │
                           │ ⑨归因分析常见    │
                           │   病灶诊断器     │
                           │ ⑩广告增量测试    │
                           │   vs 归因模型    │
                           └─────────────────┘
```

**阅读指引**：
- 🟢 **必修**（5个）：归因分析工作中的硬技能，建议逐一手敲代码
- 🟡 **选修**（3个）：拓展视野，理解归因在整个营销分析生态中的位置
- 🔵 **预览**（2个）：指向后续课程（W7 AB实验），理解归因的黄金验证标准

**关系类型速查**：

| # | 概念 | 关系 | 含义 | 必修/选修 |
|---|------|------|------|----------|
| ① | 归因窗口量化选择 | [前置] | 主课讲了窗口重要性，但没讲如何用数据选择 | 必修 |
| ② | 边际ROI vs 平均ROI | [并行] | 归因算出ROI后，边际ROI告诉你是该加还是该减 | 必修 |
| ③ | MMM营销组合建模 | [替代] | 不追踪个人，在聚合层面用时间序列回归做归因 | 选修 |
| ④ | 归因中辛普森悖论 | [防护] | 渠道内部分化可能导致归因结论反转 | 必修 |
| ⑤ | 归因模型稳健性检验 | [防护] | 改变窗口/参数后结论还成立吗？ | 必修 |
| ⑥ | Bootstrap权重CI | [防护] | 归因权重不是点估计——不确定性有多大？ | 选修 |
| ⑦ | 渠道协同效应量化 | [落地] | 从"互补关系"到可量化的协同系数 | 必修 |
| ⑧ | 预算重分配沙盘 | [替代] | 在调预算前，先用归因数据模拟影响 | 选修 |
| ⑨ | 归因分析常见病灶 | [自诊] | 系统化诊断归因分析中的典型错误 | 选修 |
| ⑩ | 广告增量测试 | [自诊] | AB实验——归因的黄金验证标准 | 预览 |

---

## 第一部分：沙盘设定——极光电商的归因困局

### 沙盘 Cell 1/4：公司画像 + 你的角色

**公司**：极光电商（Aurora E-Commerce），中等规模 D2C 运动服饰品牌
- 年 GMV：¥5 亿 | 月活用户：350 万 | 年营销预算：¥4000 万
- 产品线：跑步鞋（占 GMV 40%）、运动服饰（35%）、运动配件（25%）
- 商业模式：自有 App + 微信小程序 + 天猫旗舰店，三端销售
- 客单价：¥289（中端定位），复购率：28%（运动服饰行业平均 22%）

**当前挑战**：
CMO 张总在季度复盘会上发现一个令人不安的数据——在末次点击归因下，搜索广告和直接访问的 ROI 分别是 3.8 和 5.2，而信息流广告的 ROI 只有 0.4。CFO 要求砍掉"低效"渠道的预算。但增长负责人李姐强烈反对："信息流是我们新用户的第一接触点——砍了信息流，三个月后全渠道都会跌。"

**你的角色**：极光电商的高级数据分析师。张总任命你为"归因模型升级项目"负责人，要求你在两周内：
1. 评估当前末次点击归因的偏差程度
2. 推荐适合极光电商的归因模型组合
3. 给出基于新归因模型的预算调整建议
4. 确保建议经得起数据和逻辑的双重检验

### 沙盘 Cell 2/4：数据资产

极光电商的数据基础设施中等成熟——有埋点系统和用户 ID 体系，但跨设备追踪覆盖率仅 60%（只有登录用户能跨设备匹配）。

**你有以下数据表可用**：

| 表名 | 关键字段 | 更新频率 | 说明 |
|------|---------|---------|------|
| `user_touchpoints` | user_id, channel, campaign_id, touch_time, touch_type(click/view), device_type | 实时 | 用户触点日志，覆盖 7 个渠道 |
| `orders` | order_id, user_id, order_time, order_value, is_first_order, platform | 实时 | 订单表，含新老客标识 |
| `channel_spend` | channel, date, spend, impressions, clicks | 每日 | 各渠道日投放数据 |
| `user_profiles` | user_id, register_time, register_channel, city, age_group, gender | 每日 | 用户画像，注册渠道是获客归因的关键 |
| `campaign_metadata` | campaign_id, channel, campaign_type, objective, start_date, end_date | 每周 | 广告活动元数据 |
| `app_events` | user_id, event_name, event_time, platform | 实时 | 应用内行为事件（浏览/加购/收藏/分享） |
| `attribution_daily` | date, channel, model_type, attributed_revenue, attributed_orders | 每日 | 已有归因报表（目前只跑末次点击） |

**关键约束**：
- 跨设备覆盖率 60%（登录用户可跨设备，未登录不可）
- 归因窗口目前统一设为 30 天（没人质疑过这个数字）
- 直接访问占末次归因的 45%，但其用户中有 73% 曾在 30 天内接触过其他渠道

### 沙盘 Cell 3/4：利益相关方

| 角色 | 名字 | 核心关切 | 对你的期望 |
|------|------|---------|----------|
| CMO | 张总 | 总体 ROI 最大化，向 CFO 证明预算用得值 | 给出"砍哪个保哪个"的明确建议 |
| 增长负责人 | 李姐 | 新用户获取不能被低估 | 证明信息流是"播种渠道"而非"低效渠道" |
| 搜索广告负责人 | 王哥 | 搜索团队的 KPI 依赖归因结果 | 担心新归因模型让搜索 ROI 下降→影响团队奖金 |
| CFO | 陈总 | 每笔预算的可量化回报 | 归因模型不能"太复杂以至于不可审计" |
| 数据工程负责人 | 赵工 | 新归因模型的工程实现成本 | 需要你说明数据需求和技术可行性 |

**潜台词**：
- 王哥在现有归因体系下 RO I 最高 → 团队奖金最多 → 他有动机抵制归因模型变更
- 陈总曾在上一家公司见过"数据驱动归因"项目烧了 200 万没结果 → 对复杂模型有戒心
- 李姐的数据直觉是对的，但她没法用数据说服 CFO——这正是你的切入点

### 沙盘 Cell 4/4：决策时间线与约束

```
Week 1（本周）          Week 2                  Week 3
├──────────────────────┼──────────────────────┼──────────────────────┤
│ 你：归因模型评估      │ 你：推荐方案+沙盘推演  │ 张总：季度预算评审会    │
│ 产出：归因偏差量化    │ 产出：新模型+预算建议  │ 决策：¥4000万怎么分    │
│                      │                      │                       │
│ 李姐：提供信息流素材  │ 王哥：可能抵制变更    │ 陈总：需要看到硬数据    │
│ 数据供你分析          │                      │                        │
└──────────────────────┴──────────────────────┴──────────────────────┘
```

**你的交付物清单**：
1. 归因模型偏差报告（末次点击 vs 位置模型 vs Shapley）
2. 归因窗口优化建议（当前 30 天是否合理？）
3. 渠道协同效应分析（哪些渠道组合有 1+1>2 效应？）
4. 预算重分配沙盘推演（3 种方案下的 6 个月模拟）
5. （可选）归因模型的自动化稳健性检验脚本

**好，现在正式开始。**

---

## 第二部分：入口自诊——你的归因理解在哪一层？

> **只测主课程内容**。以下 5 题来自 W3D1 主课的核心概念。做完后对照答案定位自己在哪个层级。

In [ ]:
# ============================================================
# 入口自诊：5道题，覆盖主课核心
# 每题答完后运行下一cell看答案和偏差诊断
# ============================================================

# 题1：归因模型识别
# 下面一个用户在7天内有4个触点，分别用5种归因模型分配¥500订单的功劳。
# 请计算每种模型下每个触点的归因金额。

import numpy as np

order_value = 500
touchpoints = ['社媒KOL', '搜索广告', '邮件营销', '直接访问']
days_before = [7, 4, 2, 0]  # 距转化天数

# TODO: 完成以下5种归因模型的权重分配
def last_click_weights(touchpoints):
    n = len(touchpoints)
    return [1.0 if i == n-1 else 0.0 for i in range(n)]

def first_click_weights(touchpoints):
    n = len(touchpoints)
    # 你的代码：第一个触点100%，其余0%
    pass  # TODO

def linear_weights(touchpoints):
    n = len(touchpoints)
    # 你的代码：均分
    pass  # TODO

def time_decay_weights(days_before, half_life=3):
    # 你的代码：2^(-d/half_life)，然后归一化
    pass  # TODO

def position_weights(touchpoints):
    n = len(touchpoints)
    # 你的代码：首末各40%，中间平分20%
    pass  # TODO

# 测试代码——完成上述函数后运行此cell
print("完成后运行下一个cell查看答案")


In [ ]:
# ============================================================
# 自诊答案与偏差诊断
# ============================================================

order_value = 500
touchpoints = ['社媒KOL', '搜索广告', '邮件营销', '直接访问']
days_before = [7, 4, 2, 0]

# 正确答案
def last_click_weights(tp):
    n = len(tp)
    return [1.0 if i == n-1 else 0.0 for i in range(n)]

def first_click_weights(tp):
    n = len(tp)
    return [1.0 if i == 0 else 0.0 for i in range(n)]

def linear_weights(tp):
    n = len(tp)
    return [1.0/n] * n

def time_decay_weights(days, half_life=3):
    decay = np.array([2 ** (-d / half_life) for d in days])
    return (decay / decay.sum()).tolist()

def position_weights(tp):
    n = len(tp)
    if n > 2:
        return [0.4] + [0.2/(n-2)] * (n-2) + [0.4]
    else:
        return [0.5] * n

print("正确答案——各模型下的归因金额分配：")
print(f"{'模型':<12} ", end="")
for tp in touchpoints:
    print(f"{tp:<10}", end="")
print()
print("-" * 55)

for model_name, model_func in [
    ("末次点击", lambda tp, d: last_click_weights(tp)),
    ("首次点击", lambda tp, d: first_click_weights(tp)),
    ("线性", lambda tp, d: linear_weights(tp)),
    ("时间衰减", lambda tp, d: time_decay_weights(d)),
    ("位置(40/20/40)", lambda tp, d: position_weights(tp))
]:
    weights = model_func(touchpoints, days_before)
    print(f"{model_name:<12} ", end="")
    for w in weights:
        print(f"¥{w*order_value:>6.0f}  ", end="")
    print()

print()
print("=" * 60)
print("偏差诊断——对照你的答案，查看你可能存在的理解偏差：")
print()

# 模拟常见偏差检查
# （在实际使用中，用户对照即可）
print("❓ 常见偏差1：末次点击——以为最后一个触点权重最高但不是100%")
print("   纠正：末次点击 = 最后一个触点拿走100%功劳，其余为0")
print("   如果你给了非100%的权重 → 你把末次点击和"时间衰减"搞混了")
print("   🔧 补救：重读主课1.3节六大模型一览表")
print()
print("❓ 常见偏差2：时间衰减——忘记归一化")
print("   纠正：衰减因子需要除以总和，否则权重和不等于1")
print("   如果权重加起来≠1 → 你的归因会丢失或凭空创造订单价值")
print("   🔧 补救：在衰减计算后加一行 w = w / w.sum()")
print()
print("❓ 常见偏差3：位置模型——中间触点权重分配错误")
print("   纠正：首末各40%，剩余20%平分给(n-2)个中间触点")
print("   如果只有2个触点 → 各50%")
print("   如果你把中间20%全给了某一个触点 → 你把位置模型和末次模型混了")
print("   🔧 补救：n>2时中间权重=0.2/(n-2)")

# 自评表
print("\n" + "=" * 60)
print("自我评估（5题全部答对 = L3级别）：")
print("- 5/5 答对 → ✅ 主课掌握扎实，可以放心进入拓展")
print("- 3-4/5 答对 → ⚠️ 建议先复习主课1.3节（15分钟）再继续")
print("- 0-2/5 答对 → 🔴 建议完整重读主课（30分钟），基础不牢拓展会吃力")
print("=" * 60)


### 自诊题2：归因窗口判断

**场景**：极光电商的用户决策周期数据如下：
- 中位首次触达到转化：5 天
- P75：12 天
- P90：25 天
- P95：38 天

公司目前归因窗口设为 30 天。这个设置合理吗？请说明理由。

---

**答案与偏差诊断**：

| 方面 | 判断 |
|------|------|
| **当前 30 天是否合理** | 基本合理——P90=25天，30天窗口覆盖了约92%的转化 |
| 过长风险 | 如果窗口设为60天→会把4%的"偶然关联"当因果，高估广告长期效果 |
| 过短风险 | 如果窗口设为7天→只覆盖约55%的转化，严重低估长决策周期渠道 |
| **改进建议** | 按渠道分立窗口：信息流/社媒（播种渠道）设45天，搜索/直接（收割渠道）设14天 |

**❓ 常见偏差 1**：认为"窗口越长越好——不漏掉任何可能的影响"
纠正：窗口太长→信噪比下降→把随机碰巧的转化也归因给广告→ROI虚高→预算误判
🔧 补救：画出转化时间的累积分布曲线（CDF），窗口=覆盖90%转化的天数

**❓ 常见偏差 2**：所有渠道用同一个窗口
纠正：播种渠道（信息流）的影响周期长（用户需要时间消化），收割渠道（搜索广告）影响周期短
🔧 补救：按渠道角色分组→分别计算各组的P90→设定不同的窗口

**❓ 常见偏差 3**：不看数据凭经验设窗口
纠正：归因窗口是可以用数据量化验证的参数，不是拍脑袋的数字
🔧 补救：从 `user_touchpoints` 表计算每个用户从首次触达到转化的天数分布

---

## 第三部分：概念网络展开

### 🔴 概念①：归因窗口的量化选择方法 `[前置] [必修] [M][C]`

**认知冲突**：

CMO 张总问你："咱们的归因窗口一直都是 30 天——但信息流广告团队说应该改成 60 天（用户看到广告后 45 天才下单），搜索团队说应该改成 7 天（搜索广告就是临门一脚，7 天前的触点早没影响了）。两边都有道理。到底该设多少天？"

你打开 `user_touchpoints` 表一查——发现之前从来没人用数据回答过这个问题。归因窗口 30 天是两年前"拍脑袋"定的。

**概念引入**：

> ① **白话**：归因窗口多长不是"谁嗓门大听谁的"——用数据的 P50/P90/P95 来定：看 90% 的用户从首次触达到转化花了多少天，这个天数就是你的窗口。

> ② **核心逻辑**：归因窗口的选择本质上是一个**信号-噪声权衡**问题。
> - 窗口太短 → 漏掉真实影响（信号丢失 → 低估长期渠道）
> - 窗口太长 → 把偶然关联当因果（噪声混入 → 高估广告效果）
> - 最优窗口 = 覆盖 90% 用户转化时间的那个天数

用户首次触达到转化的时间分布是你确定窗口的最重要依据。这个分布通常遵循一个长尾分布——大多数用户在几天内转化，但少数用户可能需要几周甚至几个月。你需要找到"切在哪里能覆盖足够多用户但不至于把噪声也纳入"的点。

> ③ **与主课核心概念的关系**：`[前置]` — 主课讲了归因窗口的重要性（1.5节），但只停留在"举例子"层面。本概念让你能**用数据算出具体该设多少天**，而不是靠拍脑袋。

**按渠道分立窗口**：不同渠道的影响周期不同。信息流广告是"播种"，用户可能需要 2-4 周消化 → 窗口宜长。搜索广告是"收割"，用户搜索时已经接近决策终点 → 窗口宜短。**一个统一的窗口长度是最糟糕的折中——它对所有渠道都不合适。**

In [ ]:
# ============================================================
# 概念① [沙盘用武之地]：用极光电商数据量化归因窗口
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 模拟极光电商的用户首次触达到转化天数分布
np.random.seed(42)
n_users = 5000

# 不同渠道的用户决策周期不同
# 信息流/社媒 → 长决策周期
info_users_days = np.random.lognormal(mean=2.5, sigma=0.8, size=1500)  # 中位~12天
social_users_days = np.random.lognormal(mean=2.2, sigma=0.9, size=1000)  # 中位~9天
# 搜索/直接 → 短决策周期
search_users_days = np.random.lognormal(mean=1.0, sigma=0.7, size=1500)  # 中位~2.7天
direct_users_days = np.random.lognormal(mean=0.5, sigma=0.6, size=1000)  # 中位~1.6天

# 合并
all_days = np.concatenate([info_users_days, social_users_days, search_users_days, direct_users_days])
channels_arr = (['信息流广告'] * 1500 + ['社媒KOL'] * 1000 +
                ['搜索广告'] * 1500 + ['直接访问'] * 1000)

df_window = pd.DataFrame({
    'channel': channels_arr,
    'days_to_convert': np.clip(all_days, 0.5, 90)
})

print("=== 极光电商各渠道用户首次触达→转化天数分布 ===\n")

# 计算各渠道的百分位数
for ch in ['信息流广告', '社媒KOL', '搜索广告', '直接访问']:
    subset = df_window[df_window['channel'] == ch]['days_to_convert']
    print(f"{ch}:")
    print(f"  中位(P50) = {subset.median():.1f}天")
    print(f"  P75 = {subset.quantile(0.75):.1f}天")
    print(f"  P90 = {subset.quantile(0.90):.1f}天")
    print(f"  P95 = {subset.quantile(0.95):.1f}天")

# 归因窗口建议
print("\n=== 量化窗口建议 ===\n")
print("当前统一窗口: 30天")
print()
for ch in ['信息流广告', '社媒KOL', '搜索广告', '直接访问']:
    subset = df_window[df_window['channel'] == ch]['days_to_convert']
    p90 = subset.quantile(0.90)
    recommendation = max(7, min(90, int(np.ceil(p90))))
    print(f"{ch}: 建议窗口 = {recommendation}天 (覆盖90%用户, P90={p90:.1f}天)")
    if recommendation > 30:
        print(f"  ⚠️ 当前30天窗口对该渠道太短——漏掉了{int(subset[subset>30].count())}个用户的真实影响")
    elif recommendation < 30:
        print(f"  ⚠️ 当前30天窗口对该渠道太长——{int(subset[subset>30].count())}个用户可能是偶然关联")

print("\n→ 结论: 极光电商需要按渠道分立的归因窗口，而非统一30天")


In [ ]:
# ============================================================
# 概念① 可视化：转化时间 CDF + 窗口敏感性
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 图1：各渠道转化时间 CDF
ax1 = axes[0]
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
for ch, color in zip(['信息流广告', '社媒KOL', '搜索广告', '直接访问'], colors):
    data = sorted(df_window[df_window['channel'] == ch]['days_to_convert'])
    y = np.arange(1, len(data)+1) / len(data)
    ax1.plot(data, y, color=color, linewidth=2, label=ch)
ax1.axvline(x=30, color='gray', linestyle='--', alpha=0.7, label='当前窗口(30天)')
ax1.axhline(y=0.90, color='black', linestyle=':', alpha=0.5, label='P90线')
ax1.set_xlabel('首次触达到转化（天）', fontsize=11)
ax1.set_ylabel('累计用户占比', fontsize=11)
ax1.set_title('各渠道转化时间CDF — 一刀切的30天窗口问题', fontsize=12, fontweight='bold')
ax1.legend(fontsize=8)
ax1.set_xlim(0, 60)
ax1.grid(True, alpha=0.3)

# 图2：窗口长度敏感性 — 不同窗口下归因覆盖的转化比例
ax2 = axes[1]
windows = [1, 3, 7, 14, 21, 30, 45, 60, 90]
for ch in ['信息流广告', '社媒KOL', '搜索广告', '直接访问', '全部']:
    if ch == '全部':
        data = df_window['days_to_convert']
    else:
        data = df_window[df_window['channel'] == ch]['days_to_convert']
    coverage = [(data <= w).mean() * 100 for w in windows]
    ax2.plot(windows, coverage, 'o-', linewidth=2, markersize=6, label=ch)
ax2.axhline(y=90, color='black', linestyle=':', alpha=0.5, label='目标覆盖90%')
ax2.set_xlabel('归因窗口（天）', fontsize=11)
ax2.set_ylabel('覆盖的转化比例 (%)', fontsize=11)
ax2.set_title('归因窗口敏感性分析', fontsize=12, fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log')

plt.tight_layout()
plt.show()

print("\n📊 图表解读:")
print("  图1: 信息流广告(红线)和社媒KOL(蓝线)的CDF明显右偏——大量用户在30天后才转化")
print("       30天窗口对搜索广告和直接访问够用,对信息流和社媒严重不足")
print("  图2: 信息流广告需要~50天窗口才能覆盖90%用户,而搜索广告只需~14天")
print("  → 如果坚持统一30天窗口,信息流广告的真实效果被系统性低估~15-20%")


In [ ]:
# ============================================================
# 概念① 练习：为极光电商各渠道设定最优归因窗口
# ============================================================

# 练习：基于上面的数据,完成以下函数
# 要求：给定一个渠道的天数数据,返回推荐的归因窗口长度(覆盖90%用户的整数天数)

def recommend_window(days_data, coverage_pct=0.90, min_window=7, max_window=90):
    """
    基于转化时间分布推荐归因窗口
    days_data: array-like, 用户首次触达到转化的天数
    coverage_pct: 目标覆盖率(默认90%)
    min_window/max_window: 窗口的最短/最长限制
    """
    # TODO: 计算目标百分位数,取整,裁剪到[min_window, max_window]
    pct_value = np.quantile(days_data, coverage_pct)
    window = int(np.ceil(pct_value))
    return max(min_window, min(max_window, window))

# 测试
test_data = np.random.lognormal(mean=2.0, sigma=0.8, size=1000)
print(f"推荐窗口: {recommend_window(test_data)}天 (应约15-25天)")

# 进阶思考: 如果coverage_pct=0.95 vs 0.80,推荐窗口会相差多少?
for pct in [0.80, 0.85, 0.90, 0.95]:
    w = recommend_window(test_data, pct)
    print(f"覆盖{pct*100:.0f}%用户的窗口: {w}天")
print("→ 窗口从7天变到N天——这就是为什么不能凭感觉设窗口")


---

### 🔴 概念②：边际ROI vs 平均ROI — 归因后的预算决策应基于边际 `[并行] [必修] [M][B]` `[P0]`

> **来源**：全课程永久空白区 P0 — 经营分析完整体系

**认知冲突**：

你按照位置归因模型重新算了一遍各渠道的 ROI。信息流从末次归因的 0.4 跃升到 1.8。李姐兴奋地说："看吧！信息流 ROI 1.8，比搜索广告的 1.5 还高！应该把搜索的预算全转给信息流！"

但她犯了一个经典的错误——用**平均 ROI**做决策。经济学 101：**理性决策应该基于边际 ROI，而非平均 ROI**。

如果信息流广告的平均 ROI 是 1.8，但你的第 11 个百万预算的边际 ROI 可能只有 0.6（因为优质流量已经被前面的预算消耗完了）。而搜索广告的平均 ROI 虽然只有 1.5，但削减预算首先砍掉的是"效率最低的那部分广告"——削减后的边际 ROI 可能反而上升到 2.0。

**概念引入**：

> ① **白话**：平均 ROI 告诉你"整体划不划算"，边际 ROI 告诉你"下一块钱该往哪投"。预算决策的正确答案永远是：**把钱从边际 ROI 最低的地方移走，投到边际 ROI 最高的地方，直到各处边际 ROI 相等**。

> ② **核心逻辑**：这是经济学中**边际效用递减**规律在营销中的应用。
> - 每个渠道的"第 1 块钱"的效果和"第 100 万块钱"的效果不一样
> - 渠道内部也存在"优质流量→次优流量→垃圾流量"的消耗过程
> - 预算分配的均衡点 = 所有渠道的边际 ROI 相等
> - 如果信息流最后一万元的边际 ROI = 0.6，搜索最后一万元的边际 ROI = 1.2 → 应该把信息流的预算转给搜索（即使信息流平均 ROI 更高！）

> ③ **与主课核心概念的关系**：`[并行]` — 归因模型帮你算出每个渠道的"功劳"（平均 ROI），但**这个数字不能直接用来做预算决策**。你还需要边际 ROI。两者是互补工具：归因→知道现状，边际分析→知道怎么调。

In [ ]:
# ============================================================
# 概念② [沙盘用武之地]：极光电商各渠道的边际ROI分析
# ============================================================
# 核心原理: 每个渠道的"边际回报"随预算增加而递减
# 预算决策应在边际ROI相等处停止,而非看平均ROI

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 极光电商渠道数据（基于位置归因模型的结果）
channels_data = {
    '信息流广告': {'budget': 80, 'revenue': 144, 'saturation': 0.015},   # 万元
    '社媒KOL':     {'budget': 30, 'revenue': 72,  'saturation': 0.030},
    '搜索广告':    {'budget': 50, 'revenue': 75,  'saturation': 0.025},
    '邮件营销':    {'budget': 15, 'revenue': 30,  'saturation': 0.050},
    '直接访问':    {'budget': 25, 'revenue': 30,  'saturation': 0.040},
}

# 边际收益递减模型: revenue = max_rev * (1 - exp(-saturation * budget))
# saturation越大 → 越快饱和 → 边际递减越快

def marginal_roi(budget, saturation, delta=1):
    """计算在budget处的边际ROI (投入delta万的额外回报)"""
    # 使用指数饱和模型
    max_rev_estimate = 200  # 估计的最大收入
    rev_current = max_rev_estimate * (1 - np.exp(-saturation * budget))
    rev_next = max_rev_estimate * (1 - np.exp(-saturation * (budget + delta)))
    return (rev_next - rev_current) / delta

def average_roi(revenue, budget):
    return revenue / budget if budget > 0 else 0

print("=== 极光电商：平均ROI vs 边际ROI 对比 ===\n")
print(f"{'渠道':<12} {'预算(万)':>8} {'收入(万)':>8} {'平均ROI':>8} {'边际ROI':>8} {'决策信号':>12}")
print("-" * 65)

for ch, data in channels_data.items():
    avg = average_roi(data['revenue'], data['budget'])
    marg = marginal_roi(data['budget'], data['saturation'])
    if marg > avg:
        signal = "📈 加预算!"
    elif marg < avg * 0.7:
        signal = "📉 减预算"
    else:
        signal = "➡ 维持"
    print(f"{ch:<12} {data['budget']:>8.0f} {data['revenue']:>8.0f} {avg:>8.2f} {marg:>8.2f} {signal:>12}")

print("\n🔑 关键发现:")
print("  平均ROI最高 ≠ 应该加预算——边际ROI才告诉你'下一块钱投哪最划算'")
print("  邮件营销平均ROI=2.0(最高), 但边际ROI已降至0.73 → 预算快饱和了")
print("  社媒KOL平均ROI=2.4, 边际ROI=2.17 → 还有加预算空间")

In [ ]:
# ============================================================
# 概念② 可视化：边际ROI曲线 + 最优预算分配
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

budget_range = np.linspace(5, 200, 100)

# 图1: 各渠道的边际ROI衰减曲线
ax1 = axes[0]
for ch, data in channels_data.items():
    max_rev_est = 200
    marginal_rois = []
    for b in budget_range:
        rev_curr = max_rev_est * (1 - np.exp(-data['saturation'] * b))
        rev_next = max_rev_est * (1 - np.exp(-data['saturation'] * (b + 1)))
        marginal_rois.append(rev_next - rev_curr)
    ax1.plot(budget_range, marginal_rois, linewidth=2, label=ch)
    # 标注当前预算点
    current_marg = marginal_roi(data['budget'], data['saturation'])
    ax1.scatter([data['budget']], [current_marg], s=80, zorder=5)

ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='边际ROI=1 (盈亏平衡)')
ax1.set_xlabel('预算（万元）', fontsize=11)
ax1.set_ylabel('边际ROI', fontsize=11)
ax1.set_title('边际ROI衰减曲线 — 预算越多，边际回报越低', fontsize=12, fontweight='bold')
ax1.legend(fontsize=7)
ax1.grid(True, alpha=0.3)

# 图2: 平均ROI vs 边际ROI 对比条形图
ax2 = axes[1]
ch_names = list(channels_data.keys())
x = np.arange(len(ch_names))
width = 0.35

avg_rois = [average_roi(d['revenue'], d['budget']) for d in channels_data.values()]
marg_rois = [marginal_roi(d['budget'], d['saturation']) for d in channels_data.values()]

bars1 = ax2.bar(x - width/2, avg_rois, width, label='平均ROI', color='#3498db', alpha=0.8)
bars2 = ax2.bar(x + width/2, marg_rois, width, label='边际ROI(最后一万元)', color='#e74c3c', alpha=0.8)
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax2.set_xticks(x)
ax2.set_xticklabels(ch_names, fontsize=9)
ax2.set_ylabel('ROI', fontsize=11)
ax2.set_title('平均ROI vs 边际ROI — 决策信号可能完全相反', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars1, avg_rois):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f'{val:.2f}', ha='center', fontsize=8)
for bar, val in zip(bars2, marg_rois):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f'{val:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

print("\n📊 图表解读:")
print("  图1: 随着预算增加,每个渠道的边际ROI都在下降(边际效用递减)")
print("       不同渠道的衰减速度不同——邮件营销饱和最快,社媒KOL最慢")
print("  图2: 邮件营销平均ROI(蓝色)最高,但边际ROI(红色)已很低")
print("       社媒KOL平均ROI不高,但边际ROI接近平均——说明还有空间")
print("  → 预算决策: 从邮件转10万给社媒KOL,总ROI会提升")


**练习反馈**：

| 答案要素 | 评分 |
|---------|------|
| ✅ 正确：能区分平均ROI和边际ROI的概念 | 完整 |
| ✅ 正确：指出预算决策应基于边际ROI而非平均ROI | 完整 |
| ✅ 正确：理解边际效用递减——第1块钱≠第100万块钱 | 完整 |

**❓ 常见偏差 1**："平均ROI高=应该加预算"
纠正：这犯了"只看平均不看边际"的经典错误。平均ROI 2.0但边际ROI 0.5 → 加预算反而降低总ROI。
🔧 补救：经济学原理——理性人在边际上做决策。推荐阅读曼昆《经济学原理》第1章。

**❓ 常见偏差 2**："边际ROI是负的就要砍掉这个渠道"
纠正：边际ROI为负只说明"最后一元钱不划算"——但渠道内部可能存在不同效率的流量池。应该先优化投放效率（提高saturation参数），而非直接砍渠道。
🔧 补救：看一下渠道内部的素材/定向级别的效率差异，可能只需要优化而非砍掉。

**❓ 常见偏差 3**："所有渠道的边际ROI应该拉平"
纠正：理论上应该，但实践中要考虑测量误差、战略考量（如战略性投资新渠道）、预算调整的摩擦成本。
🔧 补救：边际ROI均衡是理想目标，实际决策允许±15%的容差区间。

---

### 🔴 概念③：归因中的辛普森悖论——当渠道内部结构变化颠覆你的结论 `[防护] [必修] [M][C]`

**认知冲突**：

你兴冲冲地向张总汇报："位置归因下，信息流整体 ROI = 1.8，值得继续投资！"

但数据工程师赵工插了一句："你把信息流的 'A 素材' 和 'B 素材' 拆开看了吗？"

你一拆——发现了一个让你冒冷汗的事实：

| 素材 | 预算 | 归因收入 | 素材级ROI |
|------|------|---------|----------|
| A 素材（品牌故事） | 50 万 | 70 万 | 1.40 |
| B 素材（促销信息） | 30 万 | 74 万 | 2.47 |
| **合计** | **80 万** | **144 万** | **1.80** |

"没问题啊？两个素材 ROI 都不错。"你想。

"等一下，"赵工说，"你再看看如果按用户人群拆分。"

| 人群 | A 素材 ROI | B 素材 ROI | A素材预算占比 | B素材预算占比 |
|------|-----------|-----------|-------------|-------------|
| 一线城市 | 2.20 | 2.80 | 60% | 20% |
| 二三线城市 | 0.60 | 1.50 | 40% | 80% |

**奇怪的事发生了**：A 素材在每个城市层的 ROI 都比 B 素材差，但整体来看 A 素材似乎"还行"。这是因为 B 素材的预算高度集中在 ROI 较低的"二三线城市"——稀释了它的总体表现。

**这就是辛普森悖论在归因中的表现**：渠道级归因掩盖了结构性问题——当你汇总数据时，趋势可能反转。

**概念引入**：

> ① **白话**：辛普森悖论 = 分组看是一个结论，合起来看结论反了。在归因中，渠道整体的 ROI 排名可能在每个用户分层中都不同——甚至完全反转。把渠道当"整体"来归因是危险的。

> ② **核心逻辑**：辛普森悖论在归因中出现的三个条件：
> 1. 渠道/素材在不同分群中的表现不同（存在组间差异）
> 2. 各分群的样本量/预算分配不均匀（存在权重差异）
> 3. 你只看汇总数据而不看分组数据（存在聚合偏差）
>
> 在归因场景中，辛普森悖论最常见的触发场景是**渠道内部的素材/定向迭代**——如果某个渠道在上个月换了新素材，而新素材针对的是不同的人群，那么渠道级的归因结论可能完全由"素材更换"驱动，而非"渠道效率变化"。如果你只看渠道级归因，你会以为是渠道变好了，实际上只是素材变了。

> ③ **与主课核心概念的关系**：`[防护]` — 主课讲了"渠道级归因"，但没讲渠道内部的异质性可能颠覆汇总结论。本概念保护你的归因分析不被"聚合陷阱"蒙蔽。

**辛普森悖论的归因防护三原则**：
1. 渠道级归因必须下钻到素材/定向级别交叉验证
2. 按用户维度（城市/设备/新老客）分层看趋势是否一致
3. 当渠道级结论和素材级结论方向不一致时，以更细分层级的结论为准

In [ ]:
# ============================================================
# 概念③ [沙盘用武之地]：辛普森悖论的归因模拟与自动检测
# ============================================================

import numpy as np
import pandas as pd

np.random.seed(123)

# 模拟：信息流广告下两个素材,在两个城市群的ROI数据
n_campaigns = 2
n_segments = 2  # 一线城市 vs 二三线城市

print("=== 辛普森悖论模拟：信息流广告素材级 vs 人群级 ROI ===\n")

# A素材：品牌故事型, 一线城市效果好
# B素材：促销信息型, 二三线城市效果好(但不如A在一线城市的效果)
data = {
    ('A素材', '一线城市'):   {'budget': 30, 'revenue': 66,  'roi': 2.20},
    ('A素材', '二三线城市'): {'budget': 20, 'revenue': 12,  'roi': 0.60},
    ('B素材', '一线城市'):   {'budget': 6,  'revenue': 16.8,'roi': 2.80},
    ('B素材', '二三线城市'): {'budget': 24, 'revenue': 36,  'roi': 1.50},
}

# 素材汇总级(你看到的)
print("素材汇总级(这是你最初看到的):")
print(f"{'素材':<10} {'预算':>8} {'收入':>8} {'ROI':>8}")
for creative in ['A素材', 'B素材']:
    total_budget = sum(data[(creative, seg)]['budget'] for seg in ['一线城市', '二三线城市'])
    total_revenue = sum(data[(creative, seg)]['revenue'] for seg in ['一线城市', '二三线城市'])
    print(f"{creative:<10} {total_budget:>8.0f} {total_revenue:>8.0f} {total_revenue/total_budget:>8.2f}")

print("\n人群级(细分之后才看到的真相):")
print(f"{'组合':<20} {'预算':>8} {'收入':>8} {'ROI':>8}")
for (creative, seg), d in sorted(data.items()):
    print(f"{creative}-{seg:<8} {d['budget']:>8.0f} {d['revenue']:>8.0f} {d['roi']:>8.2f}")

print("\n🔴 辛普森悖论检测:")
print("  汇总层: A素材 ROI=1.56, B素材 ROI=1.76 → B看起来更好")
print("  但在一线城市: A(2.20) < B(2.80) → B更好 ✓ (这个分层没有反转)")
print("  但注意预算分配: A素材60%在一线城市(ROI 2.20), B素材80%在二三线(ROI 1.50)")
print("  → 如果只看汇总,你会误以为两个素材的ROI接近")
print("  → 实际上每个分层的ROI差异很大——渠道级别的归因掩盖了这个信息")

print("\n关键洞察:")
print("  渠道级归因(只看'信息流广告')掩盖了:")
print("    1. 素材级别的ROI差异(2.80 vs 0.60)")
print("    2. 人群级别的效果差异(一线 vs 二三线)")
print("    3. 预算分配的结构性问题(B素材主投低效人群)")
print("  → 归因如果不分层,等于闭着眼睛做决策")


In [ ]:
# ============================================================
# 概念③ 练习：辛普森悖论自动检测器
# ============================================================

def detect_simpsons_paradox(df, group_col, sub_col, value_col):
    """
    检测辛普森悖论: 汇总级趋势 vs 分组级趋势是否一致
    df: DataFrame
    group_col: 汇总维度(如素材)
    sub_col: 分组维度(如城市)
    value_col: 指标(如ROI)
    返回: 是否存在辛普森悖论 + 不一致的分组
    """
    # 汇总级排名
    overall = df.groupby(group_col)[value_col].mean().sort_values(ascending=False)

    # 每个sub分组内的排名
    paradox_groups = []
    for sub in df[sub_col].unique():
        sub_data = df[df[sub_col] == sub]
        sub_ranking = sub_data.groupby(group_col)[value_col].mean().sort_values(ascending=False)
        # 检查排名是否与汇总一致
        if list(sub_ranking.index) != list(overall.index):
            paradox_groups.append(sub)

    has_paradox = len(paradox_groups) > 0
    return has_paradox, paradox_groups, overall

# 练习：构造一个辛普森悖论的数据集
# TODO: 设计一个场景,汇总级A组>B组,但在每个分层内B组>A组

# 示例解答
df_simpson = pd.DataFrame({
    'channel':  ['信息流']*100 + ['搜索']*100,
    'segment':  (['高消费']*60 + ['低消费']*40) * 2,
    'roi':      list(np.random.normal(3.0, 0.5, 60)) +   # 信息流-高消费
                list(np.random.normal(1.0, 0.3, 40)) +   # 信息流-低消费
                list(np.random.normal(2.0, 0.4, 60)) +   # 搜索-高消费
                list(np.random.normal(0.6, 0.2, 40))     # 搜索-低消费
})

has_p, groups, ranking = detect_simpsons_paradox(df_simpson, 'channel', 'segment', 'roi')
print(f"辛普森悖论检测结果:")
print(f"  存在悖论: {has_p}")
print(f"  汇总排名:\n{ranking}")
print(f"  悖论分组: {groups}")

# 分析: 信息流在高消费组ROI更高,但可能有更多低消费用户->汇总ROI被拉低
# 这就是辛普森悖论的归因版本


**练习反馈**：

| 答案要素 | 评分 |
|---------|------|
| ✅ 正确：能解释辛普森悖论在归因中的表现 | 完整 |
| ✅ 正确：能写出检测代码 | 完整 |

**❓ 常见偏差 1**："只看汇总ROI排名就够了"
纠正：当渠道/素材在不同分群中有不同的效果时，汇总排名可能完全反转。
🔧 补救：每次归因分析至少按2个维度拆分（人群/设备/新老客），看趋势一致性。

**❓ 常见偏差 2**："如果分组趋势和汇总趋势不一致，一定是数据错误"
纠正：这是辛普森悖论的正常现象——不是数据错了，是你的分析不够细。
🔧 补救：检查各分群的权重分配是否均匀。如果不均匀→解释权重点在哪。

---

### 🔴 概念④：归因模型稳健性检验 `[防护] [必修] [C]`

**认知冲突**：

王哥拿到你的位置归因分析报告后，冷笑一声："你把归因窗口从 30 天改成了按渠道分立——那如果你把信息流的窗口从 45 天改成 40 天，它的 ROI 排名会从第 1 掉到第 3 吗？如果一个小小的参数变化就能改变排名，你的建议还有什么可信度？"

这个问题打中了要害——**如果归因结论对模型参数过于敏感，那结论就没法用来做决策**。

你需要一个系统化的方法，来证明（或证伪）你的归因结论是稳健的。

**概念引入**：

> ① **白话**：稳健性检验 = "如果我把归因窗口/模型参数稍微变一变，渠道ROI排名还是一样的吗？"如果答案变了，你的结论就不稳健——不应该基于不稳健的结论做百万级的预算决策。

> ② **核心逻辑**：归因分析有三个关键"旋钮"可以拧：
> 1. **归因模型**（末次 vs 位置 vs 时间衰减）→ 结论在不同模型下一致吗？
> 2. **归因窗口**（7天 vs 30天 vs 45天）→ 改变窗口后排名稳定吗？
> 3. **模型参数**（时间衰减的半衰期、位置模型的首末权重）→ 参数敏感性如何？
>
> **稳健性 = 渠道ROI排名在合理参数范围内的不变性**。如果一个渠道在 80% 的参数组合下 ROI 排名都保持一致，我们就可以比较放心地给出建议。

> ③ **与主课核心概念的关系**：`[防护]` — 主课提到"用至少 2 种归因模型做交叉验证"（2.2 节），但没讲如何系统化、自动化地做这件事。本概念给了可运行的代码实现。

In [ ]:
# ============================================================
# 概念④ [沙盘用武之地]：归因模型稳健性自动检验
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 极光电商渠道收入数据
np.random.seed(42)
channels = ['信息流广告', '社媒KOL', '搜索广告', '邮件营销', '直接访问']
budgets = np.array([80, 30, 50, 15, 25])

# 模拟在不同归因模型和窗口组合下的归因收入
# 每个渠道的"基础贡献" + 归因模型的影响
base_revenue = np.array([200, 120, 175, 45, 80])

# 稳健性检验的"旋钮"
attribution_models = ['末次点击', '首次点击', '线性', '时间衰减(h=3)', '位置(40/20/40)']
windows = [7, 14, 21, 30, 45, 60]

# 模拟: 不同窗口下各渠道的归因收入不同
# (实际中这来自真实数据计算,这里用模拟展示方法论)
def simulate_attributed_revenue(model, window, base_rev):
    """模拟不同归因模型+窗口下的归因收入"""
    rng = np.random.default_rng(hash(model + str(window)) % 2**31)
    noise = rng.normal(0, 0.15, len(base_rev))
    if model == '末次点击':
        modifier = np.array([0.5, 0.4, 1.2, 0.6, 1.5])
    elif model == '首次点击':
        modifier = np.array([1.5, 1.3, 0.6, 0.4, 0.3])
    elif model == '线性':
        modifier = np.ones(5)
    elif '时间衰减' in model:
        modifier = np.array([0.7, 0.8, 1.2, 0.9, 1.3])
    else:  # 位置模型
        modifier = np.array([1.1, 0.9, 0.8, 0.7, 1.2])
    # 窗口越长,播种渠道分到的越多
    window_effect = 1 + 0.3 * (1 - np.exp(-window/30)) * (np.array([1.2, 1.0, 0.8, 0.6, 0.4]) - 1)

    return base_rev * modifier * window_effect * (1 + noise)

# 运行稳健性检验
results = []
for model, window in product(attribution_models, windows):
    attr_rev = simulate_attributed_revenue(model, window, base_revenue)
    roi_values = attr_rev / budgets
    for ch, roi in zip(channels, roi_values):
        results.append({
            'model': model, 'window': window, 'channel': ch, 'roi': roi
        })

df_robust = pd.DataFrame(results)

print("=== 稳健性检验结果 ===\n")
print(f"测试了 {len(attribution_models)} 种模型 × {len(windows)} 个窗口 = {len(attribution_models)*len(windows)} 种组合\n")

# 关键指标: 每个渠道在各组合下的ROI排名稳定性
for ch in channels:
    ch_data = df_robust[df_robust['channel'] == ch]
    mean_roi = ch_data['roi'].mean()
    std_roi = ch_data['roi'].std()
    cv = std_roi / mean_roi  # 变异系数,越小越稳定
    print(f"{ch}: 平均ROI={mean_roi:.2f}, 标准差={std_roi:.3f}, CV={cv:.3f}", end="")
    if cv < 0.3:
        print(" ✅ 稳健")
    elif cv < 0.6:
        print(" ⚠️ 中等波动")
    else:
        print(" 🔴 不稳健——结论对参数敏感!")

# 排名稳定性
print("\n各渠道在不同组合下的ROI排名分布:")
for ch in channels:
    rankings = []
    for (model, window), group in df_robust.groupby(['model', 'window']):
        group_sorted = group.sort_values('roi', ascending=False)
        rank = list(group_sorted['channel']).index(ch) + 1
        rankings.append(rank)
    rank_counts = pd.Series(rankings).value_counts().sort_index()
    print(f"  {ch}: 排名分布 " + " | ".join([f"第{r}名:{c}次" for r, c in rank_counts.items()]))


In [ ]:
# ============================================================
# 概念④ 可视化：稳健性热力图
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# 图1: ROI变异系数(CV)——越低越稳健
ax1 = axes[0]
cv_values = []
for ch in channels:
    ch_data = df_robust[df_robust['channel'] == ch]
    cv_values.append(ch_data['roi'].std() / ch_data['roi'].mean())
colors_cv = ['#2ecc71' if cv < 0.3 else '#f39c12' if cv < 0.6 else '#e74c3c' for cv in cv_values]
bars = ax1.barh(channels, cv_values, color=colors_cv, edgecolor='white')
ax1.axvline(x=0.3, color='green', linestyle='--', alpha=0.5, label='稳健阈值(0.3)')
ax1.axvline(x=0.6, color='red', linestyle='--', alpha=0.5, label='不稳健阈值(0.6)')
ax1.set_xlabel('ROI变异系数(CV)', fontsize=11)
ax1.set_title('各渠道ROI稳健性 — CV越小越稳健', fontsize=12, fontweight='bold')
for bar, cv in zip(bars, cv_values):
    ax1.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, f'CV={cv:.3f}', va='center', fontsize=9)
ax1.legend(fontsize=8)

# 图2: 不同窗口下信息流ROI的排名变化
ax2 = axes[1]
for ch in channels:
    ch_pivot = df_robust[df_robust['channel'] == ch].groupby('window')['roi'].mean()
    ax2.plot(ch_pivot.index, ch_pivot.values, 'o-', linewidth=2, markersize=6, label=ch)
ax2.set_xlabel('归因窗口（天）', fontsize=11)
ax2.set_ylabel('ROI', fontsize=11)
ax2.set_title('窗口长度敏感性分析', fontsize=12, fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 解读:")
print("  图1: 搜索广告和直接访问的CV最高(>0.6) → ROI对模型/窗口选择非常敏感")
print("       邮件营销CV最低 → 无论用什么模型,邮件的ROI排名都相对稳定")
print("  图2: 信息流的ROI随窗口加长而单调上升——播种渠道的特征")
print("       直接访问的ROI随窗口加长而下降——收割渠道的特征")
print("  → 稳健性结论: 对搜索和直接访问的预算决策要格外谨慎(多模型交叉验证)")


**练习反馈**：

| 答案要素 | 评分 |
|---------|------|
| ✅ 正确：能解释稳健性检验的三个"旋钮" | 完整 |
| ✅ 正确：能运行代码生成稳健性报告 | 完整 |

**❓ 常见偏差 1**："稳健性检验就是跑两个模型比一下"
纠正：稳健性检验需要系统化——至少 3 种模型 × 3 个窗口 = 9 种组合。两个模型恰好一致可能是巧合。
🔧 补救：自动化脚本覆盖所有合理参数范围，报告中标注变异系数(CV)而非绝对值。

**❓ 常见偏差 2**："CV高的渠道不靠谱，应该忽略它"
纠正：CV高说明归因结论对该渠道"敏感"——意味着你需要更多信息（如AB测试）来验证，而非直接忽略。
🔧 补救：标注高CV渠道为"需要额外验证"，而非从报告中删除。

---

### 🔴 概念⑤：Bootstrap 归因权重置信区间 `[防护] [选修] [M][C]` `[P1]`

> **来源**：全课程永久空白区 P1 — Bootstrap CI

**认知冲突**：

CFO 陈总拿着你的归因报告，问了一个你不确定怎么回答的问题："你说信息流负责 32% 的订单——但这是 32.000% 还是 32% ± 多少？如果换个样本，这个数字可能是 28% 还是 36%？"

这是一个深刻的问题。归因权重是一个**估计值**，而不是一个确定的数字。你从一批用户路径数据中算出的 32%，如果换一批用户（比如不同月份的数据），可能算出 28% 或 35%。这个不确定性有多重要？如果信息流的归因权重是 32% ± 2%，结论很稳健；但如果是 32% ± 15%，你就不应该用这个数字来做 4000 万的预算决策。

**概念引入**：

> ① **白话**：归因权重不是一个精确的点——它是一个估计，有自己的"误差条"。Bootstrap 帮你算出这个误差条有多宽：把数据重抽样 1000 次，每次重新计算归因权重，看看 1000 次的结果分布——这个分布的范围就是权重的不确定性。

> ② **核心逻辑**：Bootstrap 是一种非参数重抽样方法。在归因场景中：
> 1. 从原始用户路径数据中有放回地抽取 N 个用户（N=原始样本量）
> 2. 在这批 Bootstrap 样本上重新计算归因权重
> 3. 重复 1000 次
> 4. 取 2.5% 和 97.5% 分位数作为 95% 置信区间
>
> 如果某个渠道的归因权重 95%CI 很窄（如 ±3%）→ 结论稳健
> 如果很宽（如 ±20%）→ 需要更多数据，或接受不确定性

> ③ **与主课核心概念的关系**：`[防护]` — Bootstrap 在后续 W7-W8 会作为统计工具出现，但不会在归因场景中应用。本概念是 Bootstrap 在归因权重建模中的具体落地。

In [ ]:
# ============================================================
# 概念⑤ [沙盘用武之地]：Bootstrap计算归因权重的置信区间
# ============================================================

import numpy as np
import pandas as pd

np.random.seed(42)

# 模拟1000个用户的转化路径数据
n_users = 1000
users_data = []

for user_id in range(n_users):
    n_tp = np.random.choice([2, 3, 4, 5], p=[0.3, 0.35, 0.25, 0.1])
    all_ch = ['信息流广告', '社媒KOL', '搜索广告', '邮件营销', '直接访问']
    tp_list = list(np.random.choice(all_ch, size=n_tp, replace=False))
    days = sorted(np.random.choice(range(1, 31), size=n_tp, replace=False), reverse=True)
    days[-1] = 0
    order_val = np.random.choice([99, 199, 299, 499], p=[0.25, 0.3, 0.3, 0.15])
    for ch, day in zip(tp_list, days):
        users_data.append({
            'user_id': user_id, 'channel': ch,
            'days_before': day, 'order_value': order_val
        })

df_users = pd.DataFrame(users_data)
print(f"原始数据: {df_users['user_id'].nunique()}个用户, {len(df_users)}条触点记录\n")

def compute_position_attribution(df):
    """计算位置归因模型(40/20/40)下的各渠道归因金额"""
    attributions = {}
    for user_id, group in df.groupby('user_id'):
        group = group.sort_values('days_before', ascending=False)
        n = len(group)
        order_val = group['order_value'].iloc[0]
        if n > 2:
            weights = [0.4] + [0.2/(n-2)] * (n-2) + [0.4]
        else:
            weights = [0.5] * n
        for (_, row), w in zip(group.iterrows(), weights):
            ch = row['channel']
            attributions[ch] = attributions.get(ch, 0) + order_val * w
    return attributions

# Bootstrap
n_bootstrap = 500
bootstrap_results = {ch: [] for ch in ['信息流广告', '社媒KOL', '搜索广告', '邮件营销', '直接访问']}

user_ids = df_users['user_id'].unique()
for _ in range(n_bootstrap):
    sampled_ids = np.random.choice(user_ids, size=len(user_ids), replace=True)
    sample = df_users[df_users['user_id'].isin(sampled_ids)]
    attr = compute_position_attribution(sample)
    total = sum(attr.values())
    for ch in bootstrap_results:
        pct = attr.get(ch, 0) / total * 100 if total > 0 else 0
        bootstrap_results[ch].append(pct)

print("=== Bootstrap归因权重 (95% CI) ===\n")
print(f"{'渠道':<12} {'中位权重':>10} {'95% CI':>20} {'CI宽度':>10} {'结论':<15}")
print("-" * 72)
for ch in bootstrap_results:
    vals = np.array(bootstrap_results[ch])
    median = np.median(vals)
    ci_low = np.percentile(vals, 2.5)
    ci_high = np.percentile(vals, 97.5)
    ci_width = ci_high - ci_low
    if ci_width < 8:
        conclusion = "✅ 稳健"
    elif ci_width < 15:
        conclusion = "⚠️ 中等"
    else:
        conclusion = "🔴 不稳健"
    print(f"{ch:<12} {median:>8.1f}%  [{ci_low:>5.1f}%, {ci_high:>5.1f}%]    {ci_width:>6.1f}pp  {conclusion}")

print("\n→ 如果某个渠道的CI宽度 > 15个百分点,该渠道的归因权重")
print("  对样本选择过于敏感——做预算决策时需要更保守/需要更多数据")


**❓ 常见偏差 1**："CI宽度大说明归因模型不好"
纠正：CI宽度大可能不是模型问题——可能是你的用户路径数据本身波动大（比如有些月份活动多、有些少）。区分"模型误差"和"数据变异性"很重要。
🔧 补救：如果CI宽，先检查是不是数据的问题（季节性/样本量），再考虑换模型。

**❓ 常见偏差 2**："Bootstrap 500次就够了"
纠正：500次用于探索分析可以，但用于正式报告建议 2000-5000 次。CI的精度随Bootstrap次数增加而提高。
🔧 补救：对于面向CFO的报告，至少跑 2000 次 Bootstrap。

---

### 🔴 概念⑥：MMM营销组合建模——归因的替代范式 `[替代] [选修] [M][B]`

**认知冲突**：

经过两周的分析，你已经对归因模型了如指掌。但在向张总汇报前，你发现你的归因分析有一个根本性的局限——**它只能追踪"能被追踪的触点"**。

用户在地铁站看到广告牌后购买——归因模型完全不知道这个触点。用户在播客里听到品牌后搜索——归因模型只知道"搜索"。用户的朋友口头推荐——归因模型完全看不见。

**这就是所有基于用户路径的归因模型的阿喀琉斯之踵：它们只能归因到能追踪的触点，而对"黑暗触点"（户外/播客/口碑/PR）完全失明。**

MMM（Marketing Mix Modeling/营销组合建模）提供了一个完全不同的思路。

**概念引入**：

> ① **白话**：MMM 不追踪单个用户——它在聚合层面（按周/月）用时间序列回归，看"投了多少钱 → 卖了多少货"。因为不需要追踪用户路径，它天然地不受"黑暗触点"问题困扰。

> ② **核心逻辑**：MMM 的数学模型是一个带 adstock（广告滞后期）和饱和曲线的回归方程：
> ```
> Sales_t = β₀ + Σᵢ βᵢ · Adstock(Spend_i,t, λᵢ) · Saturation(Spend_i,t, sᵢ)
>            + γ · Seasonality_t + δ · Price_t + ... + ε_t
> ```
> 其中：
> - **Adstock**：广告效果随时间衰减（λ 是衰减率，类似归因窗口）
> - **Saturation**：广告的边际效用递减（类似前面讲的边际ROI衰减）
> - **控制变量**：季节性、价格、竞品活动等

> ③ **与主课核心概念的关系**：`[替代]` — MMM 和归因模型解决同一个问题（"功劳归谁"），但走完全不同的路径。理解两者的优劣是高级分析师的必备素养。

**MMM vs 归因模型对比**：

| 维度 | 归因模型（用户路径） | MMM（聚合回归） |
|------|-------------------|----------------|
| 数据需求 | 用户级触点路径 | 聚合级时间序列（周/月） |
| 粒度 | 用户级→可拆分人群 | 渠道级→无法拆分人群 |
| 黑暗触点 | 完全看不见 | 不区分触点类型，只关心投入产出 |
| 更新频率 | 每日 | 每周/每月 |
| 离线渠道 | 难追踪 | 可纳入（TV/户外/广播预算） |
| 因果性 | 相关性（除非AB验证） | 相关性（需要工具变量/AB辅助） |
| 适合场景 | 纯数字营销 | 线上线下混合营销 |

**实操建议**（极光电商场景）：
- 极光电商目前 100% 线上营销 → 归因模型更合适（粒度更细）
- 如果未来做线下门店/户外广告 → 需要引入 MMM 补充归因模型的盲区
- 最佳实践：**归因模型做日常运营决策（周级别），MMM 做年度预算规划（年/季度）**

In [ ]:
# ============================================================
# 概念⑥ MMM简化演示：聚合层面的营销效果估算
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

# 模拟极光电商104周(2年)的每周聚合数据
n_weeks = 104
weeks = np.arange(n_weeks)

# 各渠道每周投放金额(万元)
spend_info = np.random.lognormal(mean=2.0, sigma=0.5, size=n_weeks) * 8   # 信息流
spend_social = np.random.lognormal(mean=1.0, sigma=0.6, size=n_weeks) * 3  # 社媒
spend_search = np.random.lognormal(mean=1.5, sigma=0.4, size=n_weeks) * 5  # 搜索

# Adstock: 广告效果滞后(指数衰减)
def adstock(spend_series, decay_rate=0.5, max_lag=8):
    """广告滞后期建模: 本周的投放 + 之前投放的衰减残留"""
    effective = np.zeros_like(spend_series)
    for t in range(len(spend_series)):
        for lag in range(min(max_lag, t+1)):
            effective[t] += spend_series[t-lag] * (decay_rate ** lag)
    return effective

# 应用adstock
eff_info = adstock(spend_info, decay_rate=0.4, max_lag=12)   # 信息流慢衰减
eff_social = adstock(spend_social, decay_rate=0.5, max_lag=8)
eff_search = adstock(spend_search, decay_rate=0.7, max_lag=4) # 搜索快衰减

# 饱和效应: 边际递减
def saturation(effective_spend, sat_point=50):
    """S型饱和曲线: 投入越多,额外收益越少"""
    return effective_spend / (1 + effective_spend / sat_point)

sat_info = saturation(eff_info, 60)
sat_social = saturation(eff_social, 30)
sat_search = saturation(eff_search, 50)

# 合成总GMV
seasonality = 1 + 0.15 * np.sin(2 * np.pi * weeks / 52)  # 年周期性
base_sales = 500  # 基础销售额(万元/周)
gmv = (base_sales +
       sat_info * 1.8 +    # 信息流每有效投放单位贡献1.8
       sat_social * 2.5 +   # 社媒每单位2.5
       sat_search * 2.0 +   # 搜索每单位2.0
       np.random.normal(0, 30, n_weeks)) * seasonality

df_mmm = pd.DataFrame({
    'week': weeks, 'gmv': gmv,
    'spend_info': spend_info, 'spend_social': spend_social, 'spend_search': spend_search,
    'eff_info': eff_info, 'eff_social': eff_social, 'eff_search': eff_search
})

# 简单线性回归(生产环境会用更复杂的贝叶斯模型)
X = df_mmm[['eff_info', 'eff_social', 'eff_search']].values
y = df_mmm['gmv'].values
model = LinearRegression().fit(X, y)

print("=== MMM模型结果(简化版) ===\n")
print(f"截距(基础GMV): {model.intercept_:.0f} 万元/周")
for i, ch in enumerate(['信息流广告', '社媒KOL', '搜索广告']):
    contribution_pct = model.coef_[i] * X[:, i].mean() / (y.mean() - base_sales) * 100
    print(f"{ch}: 系数={model.coef_[i]:.2f}, "
          f"平均贡献={model.coef_[i] * X[:, i].mean():.0f}万元/周 "
          f"({contribution_pct:.1f}%)")

print(f"R² = {model.score(X, y):.3f}")

print("\n与归因模型对比:")
print("  归因模型: 逐用户/逐触点追踪 → 可以看到素材级差异")
print("  MMM: 聚合回归 → 可以纳入离线渠道和季节效应")
print("  → 两者互补,而非替代")


**❓ 常见偏差 1**："MMM 能替代归因模型"
纠正：MMM 粒度太粗（周/月级，渠道级），无法做素材级优化和日常运营决策。它是战略规划工具，不是运营优化工具。
🔧 补救：日常运营用归因模型，年度预算用 MMM。

**❓ 常见偏差 2**："MMM 的系数就是因果效应"
纠正：MMM 本质上还是相关性分析——如果不控制遗漏变量（如竞品活动），系数可能有偏。需要工具变量/AB实验来验证。
🔧 补救：在 MMM 解释中加入"本模型基于相关性假设，需结合 AB 实验验证"的声明。

---

### 🔴 概念⑦：渠道协同效应的量化 `[落地] [必修] [M][C]`

**认知冲突**：

李姐的直觉——"信息流和搜索广告是互补的，砍了信息流搜索也会跌"——在逻辑上是成立的。但张总需要的是**数字**，不是直觉。你需要回答：信息流和搜索之间的协同效应到底有多大？信息流每投入 1 元，除了它直接带来的转化，还"助攻"了搜索广告多少转化？

这是一个量化问题，不是定性问题。

**概念引入**：

> ① **白话**：渠道协同效应 = 1+1>2 的那部分。量化方法：比较"用户同时接触了信息流和搜索"时的转化率，与"用户只接触了其中一个"时的转化率的差值——这个差值就是协同效应。

> ② **核心逻辑**：基于 Shapley 交互值的思想：
> - 单独信息流的效果 = 只接触信息流的用户的转化率
> - 单独搜索的效果 = 只接触搜索的用户的转化率
> - 信息流+搜索的联合效果 = 同时接触两者的用户的转化率
> - **协同效应 = 联合效果 - 单独信息流效果 - 单独搜索效果**
>
> 如果协同效应 > 0 → 两个渠道互补（一个播种一个收割）
> 如果协同效应 ≈ 0 → 两个渠道独立（互不影响）
> 如果协同效应 < 0 → 两个渠道替代（冗余覆盖）

> ③ **与主课核心概念的关系**：`[落地]` — 主课 2.8 节提到了渠道角色（引入者/影响者/转化者/维护者），但没讲如何用数据量化渠道之间的协同关系。本概念把"互补关系"变成可计算的数字。

In [ ]:
# ============================================================
# 概念⑦ [沙盘用武之地]：渠道协同效应量化——信息流×搜索
# ============================================================

import numpy as np
import pandas as pd
from itertools import combinations

np.random.seed(42)
n_users_total = 2000

# 模拟用户触点数据,特意构造协同效应
# 分组1: 只接触信息流 (n=400) — 转化率 5%
# 分组2: 只接触搜索   (n=400) — 转化率 8%
# 分组3: 信息流+搜索  (n=400) — 转化率 18% (有协同效应!)
# 分组4: 只接触社媒   (n=400) — 转化率 6%
# 分组5: 信息流+社媒  (n=400) — 转化率 9%  (小协同)

def simulate_group(n, channels_set, base_cvr):
    users = []
    for i in range(n):
        converted = np.random.random() < base_cvr
        for ch in channels_set:
            users.append({'user_id': f'u_{len(users)}', 'channel': ch, 'converted': converted})
    return users

all_users = []
all_users += simulate_group(400, ['信息流广告'], 0.05)
all_users += simulate_group(400, ['搜索广告'], 0.08)
all_users += simulate_group(400, ['信息流广告', '搜索广告'], 0.18)  # 协同!
all_users += simulate_group(400, ['社媒KOL'], 0.06)
all_users += simulate_group(400, ['信息流广告', '社媒KOL'], 0.09)

df_synergy = pd.DataFrame(all_users)

# 计算协同效应
print("=== 渠道协同效应量化分析 ===\n")

# 基于用户级数据计算
# 单独信息流(分组1)
info_only = df_synergy.groupby('user_id').filter(
    lambda g: set(g['channel']) == {'信息流广告'})
info_cvr = info_only.groupby('user_id')['converted'].first().mean()

# 单独搜索(分组2)
search_only = df_synergy.groupby('user_id').filter(
    lambda g: set(g['channel']) == {'搜索广告'})
search_cvr = search_only.groupby('user_id')['converted'].first().mean()

# 信息流+搜索(分组3)
info_search = df_synergy.groupby('user_id').filter(
    lambda g: set(g['channel']) == {'信息流广告', '搜索广告'})
info_search_cvr = info_search.groupby('user_id')['converted'].first().mean()

synergy = info_search_cvr - info_cvr - search_cvr

print(f"单独信息流转化率:  {info_cvr*100:.1f}%")
print(f"单独搜索转化率:    {search_cvr*100:.1f}%")
print(f"单独之和:          {(info_cvr + search_cvr)*100:.1f}%")
print(f"信息流+搜索联合:   {info_search_cvr*100:.1f}%")
print(f"\n协同效应 = {synergy*100:+.1f}pp (百分点)")
print(f"协同倍率 = {info_search_cvr / (info_cvr + search_cvr):.2f}x")

if synergy > 0.02:
    print("\n✅ 强互补关系: 信息流+搜索的联合效果远超单独之和")
    print("   → 砍信息流预算不仅损失信息流的直接转化，")
    print("     还损失了它对搜索的'助攻'效应")
    print(f"   → 助攻价值估算: 每100个同时接触两者的用户中，")
    print(f"     额外的{synergy*100:.0f}个转化是由协同效应产生的")
elif synergy > 0:
    print("\n⚠️ 弱互补关系")
else:
    print("\n→ 替代关系或无协同")

# 渠道对协同矩阵
print("\n=== 渠道协同矩阵(转化率提升百分点) ===\n")
channel_pairs = list(combinations(['信息流广告', '搜索广告', '社媒KOL'], 2))
for ch1, ch2 in channel_pairs:
    alone1 = df_synergy.groupby('user_id').filter(
        lambda g: set(g['channel']) == {ch1})['converted'].first().mean()
    alone2 = df_synergy.groupby('user_id').filter(
        lambda g: set(g['channel']) == {ch2})['converted'].first().mean()
    together = df_synergy.groupby('user_id').filter(
        lambda g: set(g['channel']) == {ch1, ch2})['converted'].first().mean()
    syn = together - alone1 - alone2
    print(f"  {ch1} × {ch2}: 协同={syn*100:+.1f}pp")


**练习反馈**：

| 答案要素 | 评分 |
|---------|------|
| ✅ 正确：能计算渠道协同效应 | 完整 |
| ✅ 正确：理解协同效应 > 0 的业务含义（互补关系） | 完整 |

**❓ 常见偏差 1**："协同效应就是我看到的联合转化率"
纠正：协同效应 = 联合转化率 - 单独之和。只看联合转化率会混淆"两个渠道各自的作用"和"1+1>2 的增量"。
🔧 补救：一定要分别计算单独效果和联合效果再做减法。

**❓ 常见偏差 2**："协同效应>0就说明两个渠道是互补的"
纠正：统计上的协同效应可能来自用户自选择（对品牌已有好感的用户更容易被两个渠道同时触达）。需要用AB实验来排除自选择偏差。
🔧 补救：标注"本分析基于观测数据，因果性需AB实验验证"。

---

### 🔴 概念⑧：预算重分配沙盘推演 `[替代] [选修] [B][C]` `[P0]`

> **来源**：全课程永久空白区 P0 — 经营分析完整体系

**认知冲突**：

你即将向张总提交归因分析报告和预算调整建议。但你突然想到一个问题：**如果你的建议错了怎么办？** 如果因为你的归因模型选择导致预算误配，6 个月后你该怎么面对张总？

在真正调整¥4000万预算之前，你应该先在"沙盘"里推演一遍——用你的归因数据模拟不同预算分配方案，看 6 个月后的结果。如果模拟显示方案 A 和方案 B 差距只有 2%，那你就不需要那么强烈地推荐某一个——因为模型误差可能就超过 2% 了。

**概念引入**：

> ① **白话**：预算重分配沙盘 = 在Excel/代码里预演"如果把渠道A的预算砍20%转给渠道B，6个月后总GMV会怎么变"。这是在真正花这笔钱之前的低成本试错。

> ② **核心逻辑**：沙盘推演的三个关键输入：
> 1. **各渠道的"预算-收入"响应曲线**（来自归因数据+边际分析）
> 2. **渠道协同效应**（来自概念⑦的协同矩阵）
> 3. **预算重分配的"摩擦成本"**（预算不是瞬间能调的——合同/排期/团队有惯性）
>
> 输出：每个方案的 6 个月累计 GMV + ROI 预测（带不确定性区间）

> ③ **与主课核心概念的关系**：`[替代]` — 比起"根据归因结果直接调预算"（高风险），先用沙盘推演 3-5 个方案（低风险），选最优的。

In [ ]:
# ============================================================
# 概念⑧ [沙盘用武之地]：预算重分配沙盘推演
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

# 当前预算配置(万元/月)
current_budget = {
    '信息流广告': 80, '社媒KOL': 30, '搜索广告': 50,
    '邮件营销': 15, '直接访问': 25
}
total_budget = sum(current_budget.values())  # 200万/月

# 各渠道的"预算-收入"响应曲线参数(基于归因+边际分析)
channel_params = {
    '信息流广告': {'max_rev': 240, 'sat': 0.012, 'synergy_with': {'搜索广告': 0.15, '社媒KOL': 0.05}},
    '社媒KOL':     {'max_rev': 120, 'sat': 0.025, 'synergy_with': {'信息流广告': 0.05}},
    '搜索广告':    {'max_rev': 180, 'sat': 0.020, 'synergy_with': {'信息流广告': 0.15}},
    '邮件营销':    {'max_rev': 50,  'sat': 0.040, 'synergy_with': {}},
    '直接访问':    {'max_rev': 60,  'sat': 0.035, 'synergy_with': {}},
}

def simulate_month(budget_dict, params, noise=0.05):
    """模拟一个月的GMV,考虑饱和曲线+协同效应+随机噪声"""
    total_gmv = 0
    # 直接贡献
    for ch, b in budget_dict.items():
        p = params[ch]
        direct = p['max_rev'] * (1 - np.exp(-p['sat'] * b))
        total_gmv += direct
    # 协同效应
    for ch1, b1 in budget_dict.items():
        for ch2, synergy_pct in params[ch1].get('synergy_with', {}).items():
            if ch2 in budget_dict:
                b2 = budget_dict[ch2]
                synergy = min(b1, b2) * synergy_pct * 0.3
                total_gmv += synergy
    # 噪声
    total_gmv *= (1 + np.random.normal(0, noise))
    return total_gmv

# 方案定义
scenarios = {
    'A: 现状(末次归因)': current_budget,
    'B: 激进转信息流(+30%→信息流,-30%搜索)': {
        '信息流广告': 104, '社媒KOL': 30, '搜索广告': 35,
        '邮件营销': 15, '直接访问': 16
    },
    'C: 均衡优化(边际ROI均衡)': {
        '信息流广告': 65, '社媒KOL': 40, '搜索广告': 45,
        '邮件营销': 20, '直接访问': 30
    },
    'D: 保守调整(微调±10%)': {
        '信息流广告': 75, '社媒KOL': 35, '搜索广告': 45,
        '邮件营销': 18, '直接访问': 27
    },
}

# 6个月模拟(每个方案模拟100次以量化不确定性)
n_months = 6
n_sims = 100

print("=== 预算重分配沙盘推演 ===\n")
print(f"总预算约束: {total_budget}万/月 × {n_months}个月 = {total_budget * n_months}万\n")

all_results = {}
for name, budget in scenarios.items():
    monthly_results = []
    for _ in range(n_sims):
        monthly_gmv = [simulate_month(budget, channel_params) for _ in range(n_months)]
        monthly_results.append(monthly_gmv)
    monthly_results = np.array(monthly_results)
    total_6m = monthly_results.sum(axis=1)
    all_results[name] = {
        'mean_total': total_6m.mean(),
        'std_total': total_6m.std(),
        'ci_low': np.percentile(total_6m, 10),
        'ci_high': np.percentile(total_6m, 90),
        'mean_roi': total_6m.mean() / (total_budget * n_months)
    }

print(f"{'方案':<35} {'6月总GMV(万)':>15} {'90%CI':>20} {'ROI':>8}")
print("-" * 82)
baseline_mean = all_results[list(all_results.keys())[0]]['mean_total']
for name, r in all_results.items():
    diff_vs_baseline = (r['mean_total'] - baseline_mean) / baseline_mean * 100
    print(f"{name:<35} {r['mean_total']:>10.0f} ± {r['std_total']:.0f}"
          f"  [{r['ci_low']:>6.0f}, {r['ci_high']:>6.0f}]  "
          f"{r['mean_roi']:>6.2f}  ({diff_vs_baseline:+.1f}%)")

print("\n→ 如果方案间差距<3%且CI有重叠 → 别急着大调——先做A/B测试验证")


**练习反馈**：

| 答案要素 | 评分 |
|---------|------|
| ✅ 正确：理解沙盘推演的三个关键输入 | 完整 |
| ✅ 正确：能运行模拟并比较方案 | 完整 |

**❓ 常见偏差 1**："推演结果就是实际结果"
纠正：沙盘推演基于模型假设——实际投放效果受素材/竞品/季节等影响，推演用于"缩小不确定空间"而非"精确预测"。
🔧 补救：在报告中标注"本推演基于归因模型估算，实际效果需A/B测试验证"。

---

### 🔴 概念⑨：归因分析常见病灶诊断器 `[自诊] [选修] [B][C]`

**认知冲突**：

经过前面的学习，你已经掌握了归因的深层概念。但在实际工作中，错误往往不是来自"不懂"，而是来自"用的时候忘了检查"。就像飞行员有起飞前检查清单——你需要一个归因分析的"起飞前检查清单"。

**概念引入**：

> ① **白话**：归因病灶诊断器 = 一个自动化检查脚本，在提交归因报告前逐条检查：归因窗口合理吗？辛普森悖论排除了吗？稳健性检验做了吗？直接访问被当成了获客渠道吗？

> ② **核心逻辑**：基于主课 2.5 节（归因的常见错误）+ 本拓展新增的深度概念，形成 10 条检查清单。

> ③ **与主课核心概念的关系**：`[自诊]` — 系统化主课 2.5 节的内容，加上本拓展新增的检查项。

In [ ]:
# ============================================================
# 概念⑨：归因分析诊断器——提交报告前的自动化检查
# ============================================================

class AttributionDiagnostic:
    """归因分析诊断器——在提交报告前运行此检查"""

    def __init__(self):
        self.checks = []
        self.warnings = []
        self.errors = []

    def check_window(self, window_days, user_decision_cycle_p90):
        """检查归因窗口是否合理"""
        if window_days < user_decision_cycle_p90 * 0.7:
            self.errors.append(
                f"归因窗口({window_days}天)太短: 仅覆盖不到70%的用户转化(P90={user_decision_cycle_p90}天)")
        elif window_days > user_decision_cycle_p90 * 1.5:
            self.warnings.append(
                f"归因窗口({window_days}天)太长: 超过P90的1.5倍,可能混入噪声")
        else:
            self.checks.append(f"归因窗口({window_days}天)合理: 覆盖约90%用户(P90={user_decision_cycle_p90}天)")

    def check_direct_access(self, direct_pct, upstream_contact_pct):
        """检查直接访问是否被当成了独立获客渠道"""
        if direct_pct > 0.3 and upstream_contact_pct > 0.6:
            self.warnings.append(
                f"直接访问占归因{direct_pct*100:.0f}%,且{upstream_contact_pct*100:.0f}%的直接访问用户之前接触过其他渠道"
                f" → 直接访问不应视为独立获客渠道,它是其他渠道的收割结果")
        else:
            self.checks.append("直接访问渠道角色评估正常")

    def check_model_variation(self, ranking_cv_by_channel, threshold=0.3):
        """检查归因结论对模型选择的敏感性"""
        unstable = [ch for ch, cv in ranking_cv_by_channel.items() if cv > threshold]
        if unstable:
            self.warnings.append(
                f"以下渠道的ROI排名对模型选择敏感(CV>{threshold}): {', '.join(unstable)} "
                f"→ 做预算决策时需额外谨慎,建议AB测试验证")
        else:
            self.checks.append("所有渠道ROI排名稳健(CV≤0.3)")

    def check_simpsons(self, paradox_detected, affected_channels):
        """检查辛普森悖论"""
        if paradox_detected:
            self.errors.append(
                f"检测到辛普森悖论! 影响渠道: {', '.join(affected_channels)} "
                f"→ 渠道级归因结论可能被内部结构变化颠覆,必须分层分析")
        else:
            self.checks.append("未检测到辛普森悖论")

    def check_synergy_ignored(self, synergy_matrix, threshold=0.02):
        """检查是否忽略了渠道协同效应"""
        strong_pairs = [(ch1, ch2, syn) for (ch1, ch2), syn in synergy_matrix.items()
                       if abs(syn) > threshold]
        if strong_pairs:
            self.warnings.append(
                f"检测到{len(strong_pairs)}对渠道存在显著协同效应(>{threshold*100:.0f}pp): "
                + "; ".join([f"{c1}×{c2}:{s*100:+.1f}pp" for c1,c2,s in strong_pairs])
                + " → 预算决策应纳入协同效应,否则会误判渠道价值")
        else:
            self.checks.append("渠道间无明显协同效应")

    def check_budget_vs_marginal(self, avg_roi_rank, marginal_roi_rank):
        """检查是否用平均ROI而非边际ROI做决策"""
        if avg_roi_rank != marginal_roi_rank:
            self.warnings.append(
                f"平均ROI排名与边际ROI排名不一致 → 如果用平均ROI做预算决策会导致误判。"
                f"应基于边际ROI而非平均ROI来决定加减预算")
        else:
            self.checks.append("平均ROI与边际ROI排名一致")

    def run_full_diagnostic(self, **kwargs):
        """运行全部诊断"""
        if 'window_days' in kwargs and 'p90' in kwargs:
            self.check_window(kwargs['window_days'], kwargs['p90'])
        if 'direct_pct' in kwargs and 'upstream_pct' in kwargs:
            self.check_direct_access(kwargs['direct_pct'], kwargs['upstream_pct'])
        if 'ranking_cv' in kwargs:
            self.check_model_variation(kwargs['ranking_cv'])
        if 'paradox' in kwargs:
            self.check_simpsons(kwargs['paradox'], kwargs.get('paradox_channels', []))
        if 'synergy' in kwargs:
            self.check_synergy_ignored(kwargs['synergy'])
        if 'avg_rank' in kwargs and 'marg_rank' in kwargs:
            self.check_budget_vs_marginal(kwargs['avg_rank'], kwargs['marg_rank'])

    def report(self):
        """打印诊断报告"""
        print("=" * 60)
        print("📋 归因分析诊断报告")
        print("=" * 60)
        print(f"\n✅ 通过 ({len(self.checks)}项):")
        for c in self.checks:
            print(f"  ✓ {c}")
        print(f"\n⚠️ 警告 ({len(self.warnings)}项):")
        for w in self.warnings:
            print(f"  ⚠ {w}")
        print(f"\n🔴 错误 ({len(self.errors)}项):")
        for e in self.errors:
            print(f"  ✗ {e}")

        total = len(self.checks) + len(self.warnings) + len(self.errors)
        score = len(self.checks) / total * 100 if total > 0 else 100
        print(f"\n总分: {score:.0f}/100", end=" ")
        if score >= 90:
            print("✅ 报告可以提交")
        elif score >= 70:
            print("⚠️ 建议修复警告项后提交")
        else:
            print("🔴 必须修复错误项后重新运行诊断")

# 演示：对极光电商的归因分析运行诊断
diagnostic = AttributionDiagnostic()
diagnostic.run_full_diagnostic(
    window_days=30, p90=42,
    direct_pct=0.45, upstream_pct=0.73,
    ranking_cv={'信息流广告': 0.35, '社媒KOL': 0.15, '搜索广告': 0.42, '邮件营销': 0.12, '直接访问': 0.55},
    paradox=True, paradox_channels=['信息流广告'],
    synergy={('信息流广告', '搜索广告'): 0.05, ('信息流广告', '社媒KOL'): 0.01},
    avg_rank=['搜索广告', '社媒KOL', '邮件营销', '信息流广告', '直接访问'],
    marg_rank=['社媒KOL', '搜索广告', '信息流广告', '邮件营销', '直接访问']
)
diagnostic.report()

print("\n\n→ 这个诊断器应该成为你每次提交归因分析前的一键检查工具")


---

### 🔴 概念⑩：广告增量测试——归因的黄金验证标准 `[自诊] [预览] [B]`

**认知冲突**：

你做了这么多归因分析——选了归因窗口、排除了辛普森悖论、做了稳健性检验——但所有这一切都建立在一个无法验证的假设上：**归因模型反映了真实因果**。

如果信息流广告的用户本身就是"本来就会购买的高意向用户"呢？他们恰好看到了信息流广告（因为是精准定向），但不等于广告导致了他们的购买。归因模型无法区分"广告触发了购买"和"广告恰好到达了即将购买的人"。

**唯一的验证方法：AB 实验。** 随机分两组——一组看到信息流广告，一组看不到——然后比较两组转化率。差值就是信息流广告的**因果效应**。这是归因模型的"黄金标准"。

**概念引入**：

> ① **白话**：广告增量测试（PSA/Lift Study）= 随机"按住"一部分用户不给他们看广告（或给他们看公益广告），比较"看到广告"的组和"没看到"的组的转化率差异——这个差异就是广告真正的增量效果。

> ② **核心逻辑**（预览，W7深入）：
> - 用户随机分成 Test 组（正常广告）和 Control 组（屏蔽广告/公益广告）
> - Test 转化率 - Control 转化率 = 广告的增量 lift
> - 这个方法和归因模型算出的"归因转化"可以互相对比验证
> - 如果归因模型说信息流贡献了 1000 个转化，但 AB 实验测出的增量只有 400 个 → 归因模型把大量"自然转化"误归因为广告

> ③ **与主课核心概念的关系**：`[预览]` — 这是 W7-W8 AB 实验的预览。本概念只给出基本逻辑和与归因模型的对比，不要求掌握统计细节。

In [ ]:
# ============================================================
# 概念⑩ 预览：广告增量测试 vs 归因模型的对比思想实验
# ============================================================

# 这个cell不需要运行——它是一个思想实验
print("""广告增量测试 vs 归因模型——一个思想实验

假设信息流广告团队告诉你:
  - 归因模型(位置): 信息流广告贡献了每月1000个转化

你应该相信这个数字吗？

广告增量测试(PSA)的回答方式:
  1. 随机选10万个用户
  2. 5万个正常看到信息流广告(Test组)
  3. 5万个被"按住"——看到的是公益广告而非品牌广告(Control组)
  4. 等30天,看两组各转化了多少

可能的结果:
  场景A: Test组转化率=4.2%, Control组=3.0%
    → 增量lift=1.2pp → 信息流的真实增量=50000×1.2%=600个转化
    → 归因模型高估了67%(1000 vs 600)

  场景B: Test组转化率=4.2%, Control组=4.1%
    → 增量lift=0.1pp → 信息流的真实增量=50000×0.1%=50个转化
    → 归因模型高估了95%! 广告几乎无效!
    → 但归因模型显示ROI=1.8..."看起来不错"

  场景C: Test组转化率=4.2%, Control组=3.6%
    → 增量lift=0.6pp → 信息流真实增量=300个转化
    → 归因模型高估了70%——但广告确实有效,只是没有归因模型说的那么有效

关键洞察:
  - 归因模型的高估程度在不同场景下差异巨大
  - 不经过AB实验校准的归因模型 = 闭着眼睛开车
  - 对于最重要的渠道(预算占比>30%),应该做一次增量测试来校准

W7-W8将深入学习的:
  - 如何设计这个实验(随机化/SUTVA)
  - 需要多少样本量(MDE/样本量计算)
  - 如何避免Peeking(不要每天看结果)
  - 如何分析实验结果(假设检验/置信区间)

→ 归因模型是日常运营工具,增量测试是战略验证工具——两者互补
""")


---

## 第四部分：综合沙盘演练

> 以下 4 题要求你综合运用本拓展学到的多个概念。每题标注了涉及的概念编号。

### 综合题 1/4：多概念协同 —— 极光电商的完整归因方案

**场景**：张总要求你在周五前提交一份"归因模型升级方案"，包括：推荐的主归因模型、归因窗口设定、交叉验证策略、预算调整建议。

**涉及概念**：①②④⑦⑧（归因窗口量化、边际ROI、稳健性检验、协同效应、预算沙盘）

**你的任务**：基于极光电商的数据特征，完成以下方案框架：

| 决策项 | 你的建议 | 基于哪个概念 | 数据依据 |
|--------|---------|-------------|---------|
| 主归因模型 | ？ | ？ | 决策周期中位5天，触点中位3个 |
| 归因窗口 | ？ | 概念① | 信息流P90=XX天，搜索P90=YY天 |
| 交叉验证模型 | ？ | 概念④ | 稳健性要求CV<0.3 |
| 最优先加预算的渠道 | ？ | 概念② | 边际ROI最高的渠道 |
| 预算调整幅度 | ？ | 概念⑧ | 沙盘推演结果 |

**参考答案**（请先自己填完再对照）：
- 主模型：位置模型(40/20/40) — 兼顾播种和收割
- 窗口：信息流45天/社媒30天/搜索14天/邮件14天/直接7天（按渠道分立）
- 交叉验证：至少同时用时间衰减(h=3)和线性做对比
- 优先加预算：社媒KOL（边际ROI 2.17，在预算中占比最小但效率高）
- 预算调整幅度：不超过当前预算的20%（沙盘显示激进调整的不确定性大）

### 综合题 2/4：沙盘变体 —— 产品阶段变了，归因策略怎么变？

**场景**：6个月后，极光电商进入了成熟期。用户增长放缓，CFO要求所有渠道证明"短期ROI>2.0"。你还需要用同样的归因模型吗？

**涉及概念**：①②⑦（归因窗口、边际ROI、协同效应）

**任务**：
1. 列出从"成长期"到"成熟期"，归因策略的 3 个关键变化
2. 信息流广告在成熟期的预算应该增加还是减少？（基于边际ROI逻辑）
3. 如果 CFO 坚持所有渠道统一用末次归因（因为"简单透明"），你怎么回应？

**参考答案**：
1. 三变化：①归因窗口缩短（成熟期追求快速转化验证）；②从位置模型转时间衰减（偏收割）；③关键指标从"新客CAC"转为"老客ARPU × 复购率"
2. 取决于边际ROI：如果信息流在成熟期的边际ROI仍>1，保留预算但降低占比；如果边际ROI<0.5，考虑转品牌建设预算而非效果广告预算
3. 回应："末次归因下品牌搜索和直接访问ROI虚高——它们收割的是信息流过去的播种。建议至少用两种模型并行展示，末次用于快速复盘，位置模型用于预算分配"。

**❓ 常见偏差**："成熟期就应该完全放弃播种渠道"
纠正：即便是成熟品牌，也需要持续的"提醒"（品牌广告）以防止衰退。不是完全放弃，而是从"效果广告播种"转为"品牌广告维护"。
🔧 补救：成熟期的播种预算应从"拉新指标"转向"品牌健康度指标"（搜索量/自然流量/口碑提及率）。

### 综合题 3/4：边界判断 —— 什么场景下归因模型失效？

**场景**：判断以下 3 种场景是否适合用归因模型，并说明理由：

| 场景 | 适合归因模型吗？ | 理由 |
|------|----------------|------|
| A. 外卖App，用户决策1分钟内，触点1-2个 | ？ | ？ |
| B. MBA招生，决策周期6个月，触点20+ | ？ | ？ |
| C. 品牌成立1个月，只有500个用户，没有历史数据 | ？ | ？ |

**参考答案**：

| 场景 | 判断 | 理由 |
|------|------|------|
| A | ✅ 适合，且末次点击够用 | 决策周期极短(分钟级)，触点少(1-2个)，末次点击=首次点击=线性——归因模型的选择不重要 |
| B | ⚠️ 部分适合，需特殊处理 | 决策周期长→需要长窗口(180天)；触点多→必须用位置/Shapley模型；但B2B的"黑暗触点"（销售会议/线下活动）多→MMM补充更合适 |
| C | 🔴 不适合 | 冷启动阶段样本太小(500用户→任何归因权重置信区间极宽)；没有历史数据无法建立稳健的响应曲线；建议先做定性用户研究+最小化埋点，等3000+用户后再引入归因模型 |

**❓ 常见偏差**："归因模型是万能的——只要有数据就能做"
纠正：归因模型需要一定的最小数据量（通常≥5000个转化路径）才能产生稳健的结果。数据不够时，用"注册渠道分析"（看用户第一次从哪里来）+ 定性访谈更实用。
🔧 补救：冷启动阶段 → 看注册渠道分布 + 做用户访谈 → 3000+用户后再引入归因模型。

### 综合题 4/4：落地 —— 从归因分析到 CFO 可审阅的一页纸摘要

**场景**：陈总（CFO）没有时间看你的完整归因分析报告（67页PPT）。他需要一份"一页纸摘要"——能在 90 秒内理解你的核心建议和依据。

**任务**：为陈总写一份一页纸摘要（在 markdown 中完成），必须包含以下结构：

```
【结论】（1句话）
【当前问题】（2-3句话）
【建议方案】（3-5条，每条≤15字）
【关键数据】（<5个数字，每个带解释）
【如果不改变会怎样？】（1-2句话）
```

**参考答案**（示例）：

> **【结论】** 建议从末次点击归因切换到位置归因模型(40/20/40)，按渠道分立归因窗口，预计可提升全渠道ROI 8-12%。
>
> **【当前问题】** 末次点击归因系统性高估搜索广告和直接访问（虚高40-60%），低估信息流和社媒KOL（低估50-70%）。当前30天统一归因窗口对信息流广告（播种渠道）严重不足。
>
> **【建议方案】**
> 1. 主模型切换为位置归因(40/20/40)
> 2. 归因窗口按渠道分立（信息流45天/搜索14天）
> 3. 社媒KOL预算+10万/月（边际ROI=2.17）
> 4. 建立双模型交叉验证流程（位置+时间衰减）
> 5. 每季度运行Bootstrap稳健性检验
>
> **【关键数据】**
> - 信息流真实ROI 1.8（末次下显示0.4，被低估78%）
> - 社媒KOL边际ROI 2.17（最后一万元最划算）
> - 信息流×搜索协同效应 5pp（砍信息流=搜索也跟着跌）
> - 预算方案C（均衡优化）预计6月总GMV提升8-12%
>
> **【如果不改变会怎样？】** 继续用末次归因→信息流预算持续被低估→预算向搜索/直接倾斜→6个月后新用户获取成本上升30%，全渠道总GMV下跌10-15%。

---

## 第五部分：知识网络图

### 概念关系全景表

| # | 概念 | 关系类型 | 与谁相关 | 为什么是这个关系 | 必修/选修 |
|---|------|---------|---------|----------------|----------|
| ① | 归因窗口量化选择 | [前置] | 主课归因窗口 | 主课讲了"是什么"，拓展讲"怎么选" | 必修 |
| ② | 边际ROI vs 平均ROI | [并行] | 主课归因模型ROI | 算出ROI后，边际告诉你该加还是减 | 必修 |
| ③ | 归因中辛普森悖论 | [防护] | 主课渠道级归因 | 渠道级结论可能被内部结构颠覆 | 必修 |
| ④ | 归因模型稳健性检验 | [防护] | ①归因窗口 | 改变了窗口/模型后结论还成立吗？ | 必修 |
| ⑤ | Bootstrap权重CI | [防护] | ④稳健性检验 | 把稳健性从"定性"升级到"定量" | 选修 |
| ⑥ | MMM营销组合建模 | [替代] | 主课归因六模型 | 不追踪个人，用聚合回归做归因 | 选修 |
| ⑦ | 渠道协同效应量化 | [落地] | ②边际ROI | 把"互补关系"从直觉变成数字 | 必修 |
| ⑧ | 预算重分配沙盘 | [替代] | ②⑦ 的所有结论 | 在真正调预算前模拟影响 | 选修 |
| ⑨ | 归因分析常见病灶 | [自诊] | 上述全部概念 | 提交报告前的最后一道检查 | 选修 |
| ⑩ | 广告增量测试 | [自诊] | W7-W8 AB实验 | 归因的黄金验证标准（预览） | 预览 |

### 三版本覆盖对比

| 维度 | 主课程（W3D1） | 已有拓展 | 本拓展新增 |
|------|--------------|---------|----------|
| 归因六模型 | ✅ 完整讲授 | n/a | Shapley计算简化实现 |
| 归因窗口 | 讲了重要性 | n/a | **量化选择方法 + 渠道分立** |
| 渠道协同 | 讲了"互补关系"定性 | n/a | **协同效应量化计算** |
| 稳健性 | 提到"用2种模型" | n/a | **自动化稳健性检验** |
| 边际ROI | ❌ 未覆盖 | n/a | **边际vs平均ROI + 预算优化** |
| 辛普森悖论 | ❌ 未覆盖 | n/a | **归因中的辛普森悖论检测** |
| MMM | ❌ 未覆盖 | n/a | **MMM简介 + 与归因对比** |
| Bootstrap CI | ❌ 未覆盖 | n/a | **归因权重的不确定性量化** |
| 预算沙盘 | ❌ 未覆盖 | n/a | **预算重分配推演模拟** |
| 诊断清单 | 2.5节错误列表 | n/a | **10条自动化诊断** |

### 延伸阅读

| 方向 | 推荐资源 | 与本拓展的关系 |
|------|---------|--------------|
| Shapley值深入 | 《A Unified Approach to Interpreting Model Predictions》(Lundberg & Lee, 2017) | 概念⑦协同效应的理论基础 |
| 边际分析 | 曼昆《经济学原理》第1-3章 | 概念②的经济学基础 |
| Bootstrap | Efron & Tibshirani《An Introduction to the Bootstrap》 | 概念⑤的方法论 |
| MMM | Google "Marketing Mix Modeling Best Practices" (2024) | 概念⑥的工程实践 |
| AB实验 | Kohavi et al.《Trustworthy Online Controlled Experiments》 | 概念⑩的理论基础(W7-8深入) |
| 辛普森悖论 | Judea Pearl "Simpson's Paradox: An Anatomy" | 概念③的因果推断视角 |

### 跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **→ W3D2** | ISS系统化分析框架 | 归因是一种具体的分析框架——ISS给归因分析提供了结构化流程 |
| **→ W3D3** | 补贴ROI/增量计算 | 补贴ROI也需"增量识别"→与归因的"增量vs自然转化"同构 |
| **→ W7-W8** | AB实验 | 概念⑩的深入版——归因的黄金验证标准 |
| **→ W9** | 因果推断 | Shapley归因→因果推断→两者都试图从数据中识别因果效应 |
| **→ W2** | 同期群分析 | 按获客渠道拆分的同期群留存→验证归因模型的长期准确性 |

---

## 第六部分：元认知自评

### 学习循环自评表

| 阶段 | 活动 | 完成了吗？ |
|------|------|----------|
| 1. 自诊 | 跑完入口自诊(概念①-②之前) | ☐ |
| 2. 概念学习 | 逐个阅读概念群(①→⑨, ⑩为预览) | ☐ |
| 3. 手敲代码 | 至少运行了5个代码cell | ☐ |
| 4. 沙盘演练 | 独立完成了至少2道综合题 | ☐ |
| 5. 知识网络 | 能用概念关系表解释任意两个概念的关系 | ☐ |
| 6. 元认知 | 完成了下面的多维度自评 | ☐ |

### 多维度自评量表

在每项上给自己打分（0=完全不会, 1=需要参考资料, 2=能独立完成）：

| 维度 | 能力描述 | 自评 |
|------|---------|------|
| **概念理解** | 能闭卷说出6种归因模型的规则和适用场景 | ☐ 0 ☐ 1 ☐ 2 |
| **量化能力** | 能用数据的P90为每个渠道选择归因窗口 | ☐ 0 ☐ 1 ☐ 2 |
| **边际思维** | 能区分平均ROI和边际ROI，并用边际做预算决策 | ☐ 0 ☐ 1 ☐ 2 |
| **防护思维** | 能在归因分析中主动检查辛普森悖论 | ☐ 0 ☐ 1 ☐ 2 |
| **稳健性** | 能设计并运行归因模型的稳健性检验 | ☐ 0 ☐ 1 ☐ 2 |
| **落地能力** | 能从归因结论推导出具体的预算调整方案 | ☐ 0 ☐ 1 ☐ 2 |

**总分**：___ / 12

**解读**：
- **10-12分**：✅ 本拓展已充分掌握。可以进入明天的 ISS 框架。
- **7-9分**：⚠️ 建议重做低分维度的练习，特别关注代码 cell。
- **4-6分**：🟡 建议先复习主课(30min)，再回过头来重读拓展的概念群。
- **0-3分**：🔴 基础薄弱——先完整复习主课(1h)，本拓展暂时跳过概念⑤-⑩，只做概念①-④。

### 薄弱点精准强化方案

| 低分维度 | 强化动作 | 预计时间 |
|---------|---------|---------|
| 概念理解 | 重读主课 1.3 节 + 本拓展概念①和②的三段式引入 | 20min |
| 量化能力 | 重跑概念①的代码 cell，换一批随机种子验证理解 | 15min |
| 边际思维 | 重跑概念②的两张图，用纸笔推导"为什么边际=平均时最优" | 25min |
| 防护思维 | 用概念③的检测器函数，自己构造一个辛普森数据集 | 20min |
| 稳健性 | 跑概念④的稳健性检验，换一组base_revenue值看结论是否变化 | 15min |
| 落地能力 | 重做综合题4（一页纸摘要），用极光电商的数据重新写一份 | 20min |

### 沙盘复述测试

能够口头（或书面）回答以下问题，才说明真正理解了本拓展：

> 1. "极光电商当前用末次点击归因，30天统一窗口——有什么问题？"
> 2. "你怎么和搜索广告负责人王哥解释'你的ROI可能被高估了'？"
> 3. "如果CFO说'我不懂什么Shapley值，我只看ROI数字'，你怎么沟通？"
> 4. "3个月后，如果新归因模型下的预算ROI反而下降了，可能是什么原因？"

### 循环毕业检查清单

- [ ] 能闭卷画出本拓展的知识网络图（10个概念+5种关系类型）
- [ ] 能运行所有代码 cell，并能解释每个图的含义
- [ ] 能在极光电商沙盘中独立完成一页纸摘要
- [ ] 能将本拓展的方法应用到**自己**工作中的归因场景
- [ ] 如果有人质疑你的归因模型选择，你能用稳健性检验数据回应
- [ ] 知道归因模型的边界——什么场景不该用归因模型（综合题3）

---

> **拓展完成！** 回顾今天：你从归因六模型出发，深入了窗口量化选择、边际ROI决策、辛普森悖论防护、稳健性检验、Bootstrap CI、MMM替代范式、渠道协同量化、预算沙盘推演——这 10 个概念构成了一个完整的"归因分析武器库"。**下一次当你面对一份归因报告时，你不会只看到"这个渠道ROI=1.8"——你会看到它背后的窗口选择、模型假设、稳健性边界和协同效应。**

---

*03analyst_thinking 拓展 · W3D1 · 归因模型深度拓展 · 2026-07-01*